# BDF column-mapping report — SINTEF / DLR / FZJ Zenodo datasets

**Purpose.** For every column in each cycler's full data file, decide whether it maps to a BDF
quantity (ontology 1.2.0) **exactly**, is **close** (no perfect BDF term), or is **none**.
Conclusions are drawn *only from the data values* (range, sign, monotonicity, reset behaviour at
step / cycle boundaries, coupling to current direction) cross-checked against the ontology
`rdfs:comment` definitions.

**Key up-front finding — capacity and energy do *not* always share the same form.**
- **BioLogic (EC-Lab):** `Q charge`/`Q discharge` reset every half-cycle (they accumulate charge-only
  through rests and across all `Ns` steps of a half-cycle, then reset when polarity flips), while
  `Energy charge`/`Energy discharge` are **cumulative over the whole test** (never reset). So the
  charge-capacity column and the charge-energy column have different forms.
- **Novonix:** `Capacity (Ah)` is a cumulative signed total (does not reset); `Energy (Wh)` resets to
  zero at **every** step. Again different forms.
- **Digatron & Landt:** capacity and energy *do* share the same form (consistent), which is itself
  evidence the mappings there are coherent.

The plots below let you verify each claim. Legend in the verdict tables:
`exact` = unambiguous BDF term · `close` = nearest BDF term, imperfect (noted) · `none` = no BDF term.


In [ ]:
import matplotlib.pyplot as plt
import polars as pl

BASE = "/Users/tom/Library/CloudStorage/Box-Box/PhD Workspace/09 - Post Processing Code/Sample_data/SINTEF"
EPS = 1e-9
pl.Config.set_tbl_rows(60)

In [ ]:
def load_csv(fn, skip, sep, enc="utf8-lossy"):
    return pl.read_csv(
        f"{BASE}/{fn}",
        skip_rows=skip,
        separator=sep,
        infer_schema_length=20000,
        encoding=enc,
        truncate_ragged_lines=True,
        ignore_errors=True,
    )


def load_biologic(fn):
    # header count is declared on line 2: "Nb header lines : N"
    with open(f"{BASE}/{fn}", encoding="latin-1", errors="replace") as f:
        head = [next(f) for _ in range(5)]
    nb = next(int("".join(c for c in l if c.isdigit())) for l in head if "header lines" in l.lower())
    return load_csv(fn, nb - 1, "\t")


def load_basytec(fn, header_idx=12):
    # whitespace-delimited; header line starts with ~ and uses spaces
    rows, header, ncol = [], None, None
    with open(f"{BASE}/{fn}", encoding="latin-1", errors="replace") as f:
        for i, line in enumerate(f):
            if i < header_idx:
                continue
            toks = line.rstrip("\n").lstrip("~").split()
            if header is None:
                header, ncol = toks, len(toks)
                continue
            if len(toks) == ncol:
                rows.append(toks)
    df = pl.DataFrame({header[j]: [r[j] for r in rows] for j in range(ncol)})
    for c in df.columns:
        as_num = df[c].cast(pl.Float64, strict=False)
        if as_num.null_count() < df.height:
            df = df.with_columns(as_num.alias(c))
    return df

In [ ]:
def diagnose(df, step_col=None, cycle_col=None, current_col=None, cols=None):
    """Return a polars table of per-column diagnostics."""
    n = df.height
    step_b = (df[step_col] != df[step_col].shift(1)) if step_col in df.columns else None
    cyc_b = (df[cycle_col] != df[cycle_col].shift(1)) if cycle_col in df.columns else None
    cur = df[current_col].cast(pl.Float64, strict=False) if current_col in df.columns else None
    recs = []
    for c in cols or df.columns:
        if c not in df.columns:
            continue
        try:
            s = df[c].cast(pl.Float64, strict=False)
        except Exception:
            recs.append(dict(col=c, kind="non-numeric"))
            continue
        if s.null_count() == n:
            recs.append(dict(col=c, kind="non-numeric"))
            continue
        d = s.diff()
        rec = dict(
            col=c,
            min=round(s.min(), 4),
            max=round(s.max(), 4),
            pct_neg=round((s < -EPS).mean(), 2),
            mono_all=bool((d.fill_null(0) >= -1e-6).all()),
        )
        if step_b is not None:
            rec["mono_in_step"] = bool((d.filter(~step_b).fill_null(0) >= -1e-6).all())
            bv = s.filter(step_b)
            rec["step_bnd_is0"] = round((bv.abs() < 1e-6).mean(), 2) if bv.len() else None
        if cyc_b is not None:
            bv = s.filter(cyc_b)
            rec["cyc_bnd_is0"] = round((bv.abs() < 1e-6).mean(), 2) if bv.len() else None
        if cur is not None:
            inc, dec = d > 1e-9, d < -1e-9
            rec["inc@I>0"] = round((inc & (cur > 1e-9)).sum() / inc.sum(), 2) if inc.sum() else None
            rec["dec@I<0"] = round((dec & (cur < -1e-9)).sum() / dec.sum(), 2) if dec.sum() else None
        recs.append(rec)
    return pl.DataFrame(recs)


def plot_cols(df, cols, time_col, current_col=None, cycle_col=None, title="", maxpts=4000):
    step = max(1, df.height // maxpts)
    d = df.gather_every(step)
    t = d[time_col].cast(pl.Float64, strict=False).to_numpy()
    fig, ax = plt.subplots(figsize=(12, 4))
    for c in cols:
        if c in d.columns:
            ax.plot(t, d[c].cast(pl.Float64, strict=False).to_numpy(), label=c, lw=0.8)
    ax.set_xlabel(time_col)
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    if current_col and current_col in d.columns:
        ax2 = ax.twinx()
        ax2.plot(t, d[current_col].cast(pl.Float64, strict=False).to_numpy(), color="gray", alpha=0.35, lw=0.6)
        ax2.set_ylabel(f"{current_col} (gray)")
    plt.tight_layout()
    plt.show()

---
## 1. BioLogic — `SINTEF__NaCR32140-MP10-04__2025-08-25__GITT_0p05C_25degC__BioLogic.mpt`

GITT, single full cycle (`half cycle` 0 = charge, 1 = discharge; `cycle number` stays 0).

In [ ]:
bio = load_biologic("SINTEF__NaCR32140-MP10-04__2025-08-25__GITT_0p05C_25degC__BioLogic.mpt")
print(bio.shape)
print([c for c in bio.columns if c.strip()])
diagnose(bio, "Ns", "cycle number", "I/mA")

**Capacity columns — reset behaviour.** `(Q-Qo)` decreases on discharge (signed net);
`Q charge` accumulates through the entire charge half-cycle and resets only when discharge begins;
`Capacity` resets at the half-cycle boundary.

In [ ]:
plot_cols(
    bio,
    ["(Q-Qo)/mA.h", "Q charge/mA.h", "Q discharge/mA.h", "Capacity/mA.h"],
    "time/s",
    "I/mA",
    title="BioLogic capacity columns vs time (gray = I/mA)",
)

**The asymmetry.** Same boundary: `Q charge` drops to 0 at charge→discharge, but
`Energy charge` holds at its maximum (cumulative). Capacity is per-half-cycle; energy is cumulative.

In [ ]:
plot_cols(bio, ["Q charge/mA.h"], "time/s", title="BioLogic Q charge (resets at half-cycle)")
plot_cols(
    bio,
    ["Energy charge/W.h", "Energy discharge/W.h"],
    "time/s",
    title="BioLogic Energy charge/discharge (cumulative, no reset)",
)
# proof Q charge holds through rests and across Ns steps within the charge half-cycle
chg = bio.filter(pl.col("half cycle") == 0)
print(
    "charge half-cycle: Ns values =",
    chg["Ns"].unique().to_list(),
    "| Q charge resets within charge phase =",
    int((chg["Q charge/mA.h"].diff() < -1e-6).sum()),
    "| rows where |I|<1e-3 (rest) =",
    chg.filter(pl.col("I/mA").abs() < 1e-3).height,
)

### BioLogic verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Ns` | instrument step id (0–7, recurs) | `step_id` | **exact** |
| `time/s` | monotonic | `test_time_second` | **exact** |
| `Ecell/V` | terminal voltage | `voltage_volt` | **exact** |
| `I/mA` | measured current | `current_ampere` (×1e-3) | **exact** |
| `(Q-Qo)/mA.h` | signed, +chg/−dch, no reset | `net_capacity_ah` | **exact** (= Q−Q0) |
| `Q charge/mA.h` | charge-only, resets per **half-cycle** (not per step, holds through rest) | `cycle_charging_capacity_ah` | **close** — data shows half-cycle reset; ontology note says EC-Lab Q charge → `step_charging_capacity_ah`; no half-cycle BDF term. NOT `charging_capacity_ah` |
| `Q discharge/mA.h` | discharge-only, half-cycle reset | `cycle_discharging_capacity_ah` | **close** (same caveat) |
| `Capacity/mA.h` | active-half-cycle throughput, resets per half-cycle | `cycle_cumulative_capacity_ah` | **close** |
| `dq/mA.h` | per-record ΔQ (sign-alternating) | — | **none** (differential) |
| `Q charge/discharge/mA.h` | signed combined per half-cycle | `net_capacity_ah`-like | **close** |
| `Energy charge/W.h` | cumulative, no reset | `charging_energy_wh` | **exact** |
| `Energy discharge/W.h` | cumulative, no reset | `discharging_energy_wh` | **exact** |
| `Energy/W.h` | signed/active energy | `net_energy_wh`-like | **close** |
| `step time/s` | resets per step | `step_time_second` | **exact** |
| `P/W` | V·I | `power_watt` | **exact** |
| `R/Ohm` | resistance | `internal_resistance_ohm` | **exact** |
| `Temperature/°C` | 22–24 °C | `ambient_temperature_celsius` | **close** (could be device temp) |
| `cycle number` | 0 here | `cycle_count` | **exact** |
| `half cycle` | 0/1 | — | **close** (`cycle_count` granularity differs) |
| `mode` | 1/2/3 technique code | — | **close** (`step_type`, numeric not label) |
| `ox/red`,`error`,`control changes`,`Ns changes`,`counter inc.`,`I Range`,`x`,`Efficiency/%`,`Capacitance charge/discharge`,`control/mA` | flags / setpoints / derived | — | **none** |

**Headline:** existing `charging_capacity_ah ← Q charge` and `discharging_capacity_ah ← Q discharge`
are wrong (those columns reset per half-cycle; the true cumulative quantity is carried by
`Energy charge`/`Energy discharge` only on the energy side). `cumulative_capacity_ah ← (Q-Qo)` is wrong
— `(Q-Qo)` is `net_capacity_ah`. `step_net_capacity_ah ← dq` is wrong — `dq` maps to nothing.

---
## 2. Digatron — `FZJ__INR21700__20250606__HPPC__25degC__Digatron.csv`

HPPC. Capacity and energy share the same form here (consistent set).

In [ ]:
dig = load_csv("FZJ__INR21700__20250606__HPPC__25degC__Digatron.csv", 0, ",")
print(dig.shape)
print(dig.columns)
diagnose(dig, "Step", "Cycle", "Current#A")

`AhAccu` is exactly `AhCha − AhDch` (signed net); `AhStep` resets each step and only ever
increases (|throughput|). Energy columns mirror this.

In [ ]:
print(
    "AhAccu == AhCha-AhDch ? max|diff| =", float((dig["AhAccu#Ah"] - (dig["AhCha#AH"] - dig["AhDch#Ah"])).abs().max())
)
print(
    "WhAccu == WhCha-WhDch ? max|diff| =", float((dig["WhAccu#Wh"] - (dig["WhCha#Wh"] - dig["WhDch#Wh"])).abs().max())
)
print("AhStep min (>=0 => throughput, <0 => net) =", float(dig["AhStep#Ah"].min()))
plot_cols(
    dig,
    ["AhCha#AH", "AhDch#Ah", "AhAccu#Ah", "AhStep#Ah"],
    "Program Duration#s",
    "Current#A",
    title="Digatron capacity columns",
)
plot_cols(
    dig,
    ["WhCha#Wh", "WhDch#Wh", "WhAccu#Wh", "WhStep#Wh"],
    "Program Duration#s",
    "Current#A",
    title="Digatron energy columns (same form as capacity)",
)

### Digatron verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Step` | instrument step id | `step_id` | **exact** (currently `step_index` — wrong) |
| `Status` | CHA/DCH/PAU string | `step_type` | **exact** |
| `Timestamp` | datetime+tz | `unix_time_second` | **exact** |
| `Program Duration#s` | monotonic | `test_time_second` | **exact** |
| `Step Duration#s` | resets per step | `step_time_second` | **exact** (synonym `Step Time` never matches — fix) |
| `Cycle` | — | `cycle_count` | **exact** |
| `AhAccu#Ah` | = AhCha−AhDch (signed) | `net_capacity_ah` | **exact** (currently `cumulative_capacity_ah` — wrong) |
| `AhCha#AH` | cumulative charge | `charging_capacity_ah` | **exact** |
| `AhDch#Ah` | cumulative discharge | `discharging_capacity_ah` | **exact** |
| `AhStep#Ah` | resets per step, ≥0, both dirs | `step_cumulative_capacity_ah` | **exact** (currently `step_net_capacity_ah` — wrong) |
| `AhBal#Ah` | ≥0, ≠ net (see plot) | — | **close / none** (no clean term; not the net) |
| `WhAccu#Wh` | = WhCha−WhDch | `net_energy_wh` | **exact** (currently `cumulative_energy_wh` — wrong) |
| `WhCha#Wh` / `WhDch#Wh` | cumulative | `charging_energy_wh` / `discharging_energy_wh` | **exact** |
| `WhStep#Wh` | resets per step, ≥0 | `step_cumulative_energy_wh` | **exact** (currently `step_net_energy_wh` — wrong) |
| `Voltage#V` / `Current#A` | — | `voltage_volt` / `current_ampere` | **exact** |
| `T1#degC` | 25–33 °C cell channel | `temperature_t1_celsius` | **exact** |
| `Tenv#degC` | ~25 °C ambient | `ambient_temperature_celsius` | **exact** (currently unmapped) |
| `Cycle Level`,`Procedure`,`Procedure Level` | metadata | — | **none** |


In [ ]:
plot_cols(
    dig,
    ["AhBal#Ah", "AhStep#Ah", "AhAccu#Ah"],
    "Program Duration#s",
    title="Digatron AhBal vs AhStep vs AhAccu (what is AhBal?)",
)

---
## 3. Novonix — `SINTEF__SLPBA842124HV-06__20241011__DCIR__0p1C__25degC__Novonix.csv`

DCIR. **Capacity is cumulative-net; Energy is per-step** (the asymmetry).

In [ ]:
nov = load_csv("SINTEF__SLPBA842124HV-06__20241011__DCIR__0p1C__25degC__Novonix.csv", 20, ",")
print(nov.shape)
print(nov.columns)
diagnose(nov, "Step Number", "Cycle Number", "Current (A)")

In [ ]:
nb_ = nov.with_columns((pl.col("Step Number") != pl.col("Step Number").shift(1)).alias("sb"))
print("Capacity |val| at step boundaries (mean) =", float(nb_["Capacity (Ah)"].filter(nb_["sb"]).abs().mean()))
print("Energy   |val| at step boundaries (mean) =", float(nb_["Energy (Wh)"].filter(nb_["sb"]).abs().mean()))
plot_cols(nov, ["Capacity (Ah)"], "Run Time (h)", "Current (A)", title="Novonix Capacity (cumulative net, no reset)")
plot_cols(nov, ["Energy (Wh)"], "Run Time (h)", "Current (A)", title="Novonix Energy (resets every step)")

### Novonix verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Date and Time` | datetime | `unix_time_second` | **exact** |
| `Cycle Number` | — | `cycle_count` | **exact** |
| `Step Number` | monotonic unique (→276) | `step_count` | **exact** |
| `Step position` | recurs 0–3 | `step_id` | **exact** (currently `step_index` — wrong) |
| `Step Type` | 0/1/2 code | `step_type` | **close** (numeric, not label) |
| `Run Time (h)` | monotonic | `test_time_second` (×3600) | **exact** |
| `Step Time (h)` | resets per step | `step_time_second` (×3600) | **exact** |
| `Current (A)` / `Potential (V)` | — | `current_ampere` / `voltage_volt` | **exact** |
| `Capacity (Ah)` | signed, no reset | `net_capacity_ah` | **exact** |
| `Energy (Wh)` | signed, resets every step | `step_net_energy_wh` | **exact** (currently `net_energy_wh` — wrong) |
| `Power(W)` | V·I | `power_watt` | **exact** |
| `Temperature (°C)` | all −9999 sentinel (no sensor) | `ambient_temperature_celsius` | **exact** mapping, **invalid data** |
| `Circuit Temperature (°C)` | 25–28 °C | `temperature_t1_celsius` | **close** (could be surface) |
| `dVdt (V/h)`, `dIdt (A/h)` | derivatives | — | **none** |

**Note the missing partner columns:** Novonix exports cumulative net *capacity* but only per-step
*energy* — there is no cumulative energy column and no per-step capacity column.

---
## 4. Landt — `SINTEF__LiGrR2032__2024-04-30__25degC__Landt.csv` (and `.txt`)

Capacity and energy both per-step (consistent).

In [ ]:
lc = load_csv("SINTEF__LiGrR2032__2024-04-30__25degC__Landt.csv", 6, ",")
print(lc.shape)
print(lc.columns)
print(
    "step_index transition sequence:",
    lc.filter(pl.col("step_index") != pl.col("step_index").shift(1))["step_index"].to_list()[:12],
)
diagnose(lc, "step_index", "cycle_index", "current_A")

In [ ]:
plot_cols(
    lc,
    ["discharge_capacity_Ah", "charge_capacity_Ah"],
    "test_time_s",
    "current_A",
    title="Landt capacities (reset per step)",
)

### Landt CSV verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `channel_index` | global row counter 1..N | `record_index` | **close** (global record counter) |
| `cycle_index` | — | `cycle_count` | **exact** |
| `step_index` | recurs 1→2→3→2 | `step_id` | **exact** (currently `step_count` — wrong) |
| `date_time_iso_string` | datetime | `unix_time_second` | **exact** (currently unmapped) |
| `test_time_s` | monotonic | `test_time_second` | **exact** |
| `step_time_s` | resets per step | `step_time_second` | **exact** |
| `current_A` / `voltage_V` | — | `current_ampere` / `voltage_volt` | **exact** |
| `discharge_capacity_Ah` | resets per step | `step_discharging_capacity_ah` | **exact** (unmapped) |
| `charge_capacity_Ah` | resets per step (0 here) | `step_charging_capacity_ah` | **exact** (unmapped) |
| `discharge_energy_Wh` | resets per step | `step_discharging_energy_wh` | **exact** (unmapped) |
| `charge_energy_Wh` | resets per step (0 here) | `step_charging_energy_wh` | **exact** (unmapped) |
| `temperature_1_C` / `_2_C` / `_3_C` | 0 (no sensor) | `temperature_t1/t2/t3_celsius` | **exact** mapping, no data |
| `step_name` | "rest" etc | `step_type` | **exact** |
| `Pressure_Psi` | 0, psi | — | **none** (psi not supported; no data) |

**Landt TXT** (same cell) analysed below — `Amp-hr`/`Watt-hr` are single signed accumulators.

In [ ]:
def load_landt_txt():
    return pl.read_csv(
        f"{BASE}/SINTEF__LiGrR2032__2024-04-30__25degC__Landt.txt",
        skip_rows=1,
        separator="\t",
        infer_schema_length=20000,
        encoding="utf8-lossy",
        truncate_ragged_lines=True,
        ignore_errors=True,
    )


lt = load_landt_txt()
print(lt.columns)
diagnose(lt, "Step", "Cyc#", "Amps")

**Landt TXT verdicts:** `Rec#`→`record_index` (close), `Cyc#`→`cycle_count`, `Step`→`step_id`
(recurs 1→2→3→2), `Test(Sec)`→`test_time_second`, `Step(Sec)`→`step_time_second`, `Amps`→`current_ampere`,
`Volts`→`voltage_volt`, `DPt-Time`→`unix_time_second`, `State`→**close** (`step_type`, single-char code),
`ES`→**none** (event/status flag).
`Amp-hr`/`Watt-hr`: data shows **≥0** (`pct_neg=0`) and **reset every step/cycle** (`cyc_bnd_is0=1.0`),
max = the CSV per-step capacity → `step_cumulative_capacity_ah` / `step_cumulative_energy_wh`
(|throughput| within step), **not** signed `step_net_*`.

---
## 5. Basytec — `DLR__LiLNMOHydra0b__20221130__GITT__25degC__Basytec.txt`

GITT. Capacity only (no energy columns). Note gravimetric `[Ah/kg]` variants.

In [ ]:
bas = load_basytec("DLR__LiLNMOHydra0b__20221130__GITT__25degC__Basytec.txt")
print(bas.shape)
print(bas.columns)
diagnose(bas, "Line", "Cyc-Count", "I[A]")

In [ ]:
plot_cols(
    bas,
    ["Ah-Charge", "Ah-Discharge", "Ah-Step"],
    "Time[h]",
    "I[A]",
    title="Basytec capacities (Ah-Charge/Discharge cumulative; Ah-Step resets per step)",
)
print("Ah-Step min (<0 => signed/net) =", float(bas["Ah-Step"].min()))

### Basytec verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Time[h]` | monotonic | `test_time_second` (×3600) | **exact** |
| `DataSet` | global counter | `record_index` | **exact** |
| `t-Set[h]` | resets per step | `step_time_second` | **close** (per-set time; add synonym) |
| `Line` | instrument step id | `step_id` | **exact** |
| `Command` | Charge/Discharge/Pause | `step_type` | **exact** |
| `U[V]` / `I[A]` | — | `voltage_volt` / `current_ampere` | **exact** |
| `Ah[Ah/kg]` | gravimetric net | — | **none** (per-mass; no BDF term) |
| `Ah-Charge` | cumulative charge | `charging_capacity_ah` | **exact** (currently unmapped) |
| `Ah-Discharge` | cumulative discharge | `discharging_capacity_ah` | **exact** (currently unmapped) |
| `Ah-Step` | resets per step, signed | `step_net_capacity_ah` | **exact** (currently unmapped) |
| `Ah-Set` | signed over "set" | `net_capacity_ah` | **close** (uncertain) |
| `Ah-Step[Ah/kg]`, `Ah-Set[Ah/kg]` | gravimetric | — | **none** |
| `T1[°C]` | channel-1 temp | `temperature_t1_celsius` | **exact** (normalizer + `basytec/url` use `ambient_temperature_celsius` — wrong) |
| `Cyc-Count` | — | `cycle_count` | **exact** |
| `State` | numeric state code | — | **none** (≈`step_type`) |

**Note:** Basytec `Ah-Step` is *signed* (min<0) → `step_net_capacity_ah`, whereas Digatron `AhStep`
is ≥0 → `step_cumulative_capacity_ah`. Same-looking header, different quantity — verified from data.

---
## 6. Neware — `.nda` (`SINTEF__G20M7…Neware.nda`, via fastnda) + xlsx sample

The `.nda` is read by `fastnda`, which already emits BDF-style names. The xlsx
(`tests/data/neware/sample_data_neware.xlsx`) is a 105-row, single-step sample.

In [ ]:
import fastnda

nda = fastnda.read(f"{BASE}/SINTEF__G20M7-202512-Gru6mV__20251228__C30__25degC__Neware.nda")
print(nda.shape)
print(nda.columns)
print(
    "cycles:",
    nda["cycle_count"].unique().to_list(),
    "| steps:",
    nda["step_index"].unique().to_list(),
    "| step_type:",
    nda["step_type"].unique().to_list(),
)
diagnose(nda, "step_count", "cycle_count", "current_mA")

`capacity_mAh` and `energy_mWh` reset at every step (`step_bnd_is0=1.0`) and are **signed by
step direction** — positive in charge steps, negative in discharge, zero in rest. That is exactly
`step_net_capacity_ah` / `step_net_energy_wh`.

In [ ]:
print(
    nda.group_by("step_type")
    .agg(
        pl.col("capacity_mAh").min().alias("cap_min"),
        pl.col("capacity_mAh").max().alias("cap_max"),
        pl.col("energy_mWh").min().alias("en_min"),
        pl.col("energy_mWh").max().alias("en_max"),
    )
    .sort("step_type")
)
plot_cols(
    nda, ["capacity_mAh"], "total_time_s", "current_mA", title="Neware NDA capacity_mAh (signed, resets per step)"
)
plot_cols(nda, ["energy_mWh"], "total_time_s", "current_mA", title="Neware NDA energy_mWh (signed, resets per step)")

### Neware NDA verdicts (fastnda output)

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `index` | global row counter | `record_index` | **close** (currently unmapped) |
| `voltage_V` | — | `voltage_volt` | **exact** |
| `current_mA` | — | `current_ampere` (×1e-3) | **exact** |
| `unix_time_s` | absolute time | `unix_time_second` | **exact** |
| `step_time_s` | resets per step | `step_time_second` | **exact** |
| `total_time_s` | monotonic | `test_time_second` | **exact** |
| `cycle_count` | — | `cycle_count` | **exact** |
| `step_count` | monotonic unique step counter | `step_count` | **exact** |
| `step_index` | 1–6, = step id (single cycle here) | `step_id` | **close** — Neware step id recurs across cycles → `step_id`, not `step_index` (can't prove recurrence in 1-cycle file) |
| `step_type` | CC_Chg/CC_DChg/CV_Chg/Rest | `step_type` | **exact** (currently unmapped) |
| `capacity_mAh` | signed, resets per step | `step_net_capacity_ah` | **exact** ✓ (NDA_NORMALIZER already correct) |
| `energy_mWh` | signed, resets per step | `step_net_energy_wh` | **exact** ✓ |

**Neware NDA mappings are already right** for capacity/energy (signed step-net). Only gaps:
`step_type` and `index`/`record_index` unmapped, and `step_index`→`step_id`.

In [ ]:
xls = pl.read_excel("tests/data/neware/sample_data_neware.xlsx")
print(xls.shape)
print(xls.columns)
diagnose(xls, "Step Index", "Cycle Index", "Current(mA)")
print(
    "xlsx step indices present:",
    xls["Step Index"].unique().to_list(),
    "| cycles:",
    xls["Cycle Index"].unique().to_list(),
)

### Neware XLSX verdicts (105 rows, single charge step — limited)

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Date` | datetime | `unix_time_second` | **exact** |
| `Time` | per-step elapsed (datetime base) | `step_time_second` | **close** |
| `Total Time` | test elapsed (datetime base) | `test_time_second` | **exact** |
| `Cycle Index` | — | `cycle_count` | **exact** |
| `Step Index` | step id (only 2 here) | `step_id` | **close** (single value; recurs in full data) |
| `Current(mA)` | — | `current_ampere` (×1e-3) | **exact** |
| `Voltage(V)` | — | `voltage_volt` | **exact** |
| `Capacity(mAh)` | active-step capacity (≥0 here; signed step-net in NDA) | `step_net_capacity_ah` | **close** (single step can't show reset/sign; NDA confirms step-net) |
| `Chg. Cap.(mAh)` | charge-only step capacity | `step_charging_capacity_ah` | **close** — Neware convention is per-step; current `charging_capacity_ah` (test-level) likely wrong. Needs multi-step xlsx to confirm |
| `DChg. Cap.(mAh)` | discharge-only step capacity (0 here) | `step_discharging_capacity_ah` | **close** (same caveat) |

**Header-match risk:** current NEWARE synonyms are `Chg.Capacity({unit})` / `Charge Capacity({unit})`,
but the xlsx headers are `Chg. Cap.(mAh)` / `DChg. Cap.(mAh)` — different punctuation/spacing, so they
may not even match. Confirm against a multi-step Neware export before fixing the target quantity.

---
## 7. Arbin — Zenodo `shandong__nacr32140-mp10__2023-10-10__pulse__25degC__arbin.CSV`

Pulse/HPPC test (NaCR32140-MP10). `Charge Capacity` / `Discharge Capacity` /
`Charge Energy` / `Discharge Energy` look cumulative at a glance, but they are
**scripted resets** tied to the test plan, not per-step or per-cycle resets
(file has a single `Cycle Index` throughout).

In [ ]:
import requests

_ARBIN_URL = (
    "https://zenodo.org/api/records/18986774/files/"
    "shandong__nacr32140-mp10__2023-10-10__pulse__25degC__arbin.CSV/content"
)


def load_arbin():
    r = requests.get(_ARBIN_URL, timeout=60)
    r.raise_for_status()
    return pl.read_csv(
        r.content, infer_schema_length=20000, encoding="utf8-lossy", truncate_ragged_lines=True, ignore_errors=True
    )


arb = load_arbin()
print(arb.shape)
print(arb.columns)
diagnose(arb, "Step Index", "Cycle Index", "Current (A)")

In [ ]:
def drops(df, col, step_col="Step Index"):
    s = df[col].cast(pl.Float64, strict=False)
    d = s.diff().fill_null(0)
    idx = (d < -1e-6).arg_true()
    return [
        (int(i), int(df[step_col][i - 1]), int(df[step_col][i]), round(float(s[i - 1]), 4), round(float(s[i]), 4))
        for i in idx
    ]


print("Charge Capacity drops (row, step_from, step_to, prev, val):")
for row in drops(arb, "Charge Capacity (Ah)"):
    print(" ", row)
print("Discharge Capacity drops (first 6):")
for row in drops(arb, "Discharge Capacity (Ah)")[:6]:
    print(" ", row)

In [ ]:
plot_cols(
    arb,
    ["Voltage (V)", "Charge Capacity (Ah)", "Discharge Capacity (Ah)"],
    "Test Time (s)",
    "Current (A)",
    title="Arbin capacities — scripted resets at steps 9/12/74, not per-step",
)

In [ ]:
plot_cols(
    arb,
    ["Charge Capacity (Ah)", "Discharge Capacity (Ah)", "Capacity (Ah)"],
    "Test Time (s)",
    "Current (A)",
    title="Arbin capacities — scripted resets at steps 9/12/74, not per-step",
)

### Arbin verdicts

| column | data behaviour | BDF mapping | grade |
|---|---|---|---|
| `Data Point` | global counter | `record_index` | **exact** |
| `Date Time` | datetime | `unix_time_second` | **exact** |
| `Test Time (s)` | monotonic | `test_time_second` | **exact** |
| `Step Time (s)` | resets per step | `step_time_second` | **exact** |
| `Cycle Index` | constant in this file (single cycle) | `cycle_count` | **exact** |
| `Step Index` | instrument step id | `step_id` | **exact** |
| `Current (A)` / `Voltage (V)` | — | `current_ampere` / `voltage_volt` | **exact** |
| `Power (W)` | — | `power_watt` | **exact** |
| `Charge Capacity (Ah)` | resets once, at step-9 boundary (end of conditioning, start of pulse loop); cumulative elsewhere | `charging_capacity_ah` | **known bug** — scripted reset, see `known_validity_bugs` |
| `Discharge Capacity (Ah)` | resets at steps 12 and 74 every pass through the pulse loop, not other step boundaries | `discharging_capacity_ah` | **known bug** — scripted reset |
| `Charge Energy (Wh)` | mirrors Charge Capacity | `charging_energy_wh` | **known bug** — scripted reset |
| `Discharge Energy (Wh)` | mirrors Discharge Capacity | `discharging_energy_wh` | **known bug** — scripted reset |
| `Capacity (Ah)` | tracks whichever of Charge/Discharge Capacity is active for the current step direction | — | **none** (redundant; derived from the two capacity columns) |
| `ACR (Ohm)` | entirely empty in this export | `ac_internal_resistance_ohm` | **none** (no data; see `known_validity_bugs`) |
| `Internal Resistance (Ohm)` | — | `dc_internal_resistance_ohm` | **exact** |
| `Aux_Temperature_1 (C)` | — | `temperature_t1_celsius` | **exact** |
| `mAh/g`, `dV/dt (V/s)`, `dQ/dV (Ah/V)`, `dV/dQ (V/Ah)`, `Aux_dT/dt_1 (C/s)`, `TC_Counter*` | derived/aux | — | **none** (no BDF term) |

**Note:** the "scripted reset" columns are correctly mapped — `assert_mono_nonneg` (global
monotonicity) is the wrong check for them, which is why `tests/integration/test_cases.py`
documents them under `known_validity_bugs` instead of fixing the mapping.

---
## 8. Summary — corrections implied (for corpus first, then normalizers)

**Wrong mappings to fix (data-verified):**

| cycler | column | current → correct |
|---|---|---|
| BioLogic | `(Q-Qo)` | `cumulative_capacity_ah` → `net_capacity_ah` |
| BioLogic | `Q charge` / `Q discharge` | `charging/discharging_capacity_ah` → **half-cycle** (`cycle_*`; no exact term) |
| BioLogic | `dq` | `step_net_capacity_ah` → **none** |
| Digatron | `Step` | `step_index` → `step_id` |
| Digatron | `AhAccu` / `WhAccu` | `cumulative_*` → `net_*` |
| Digatron | `AhStep` / `WhStep` | `step_net_*` → `step_cumulative_*` |
| Digatron | `AhBal` | `net_capacity_ah` → **none** (real net is `AhAccu`) |
| Novonix | `Step position` | `step_index` → `step_id` |
| Novonix | `Energy (Wh)` | `net_energy_wh` → `step_net_energy_wh` |
| Landt | `step_index` | `step_count` → `step_id` |
| Basytec | `T1[°C]` | `ambient_temperature_celsius` → `temperature_t1_celsius` |
| Neware | `step_index` (nda) | `step_index` → `step_id` |
| Neware | `Chg./DChg. Cap.` (xlsx) | `charging/discharging_capacity_ah` → `step_*` (per-step; confirm on multi-step file) |

**Already correct (verified):** Neware `.nda` `capacity_mAh`→`step_net_capacity_ah`,
`energy_mWh`→`step_net_energy_wh`; Digatron `AhCha/AhDch`→`charging/discharging_capacity_ah`;
BioLogic `Energy charge/discharge`→`charging/discharging_energy_wh`.

**Unmapped columns that should be added:** Digatron `Step Duration#s`→`step_time_second`,
`Tenv`→`ambient_temperature_celsius`, `Status`→`step_type`; Novonix `Step Type`→`step_type`;
Landt CSV step-level capacities/energies, temps, `date_time_iso_string`→`unix_time_second`,
`step_name`→`step_type`; Basytec `Ah-Charge`/`Ah-Discharge`/`Ah-Step`, `Command`→`step_type`;
Neware nda `step_type`, `index`→`record_index`.

**Conceptual gaps (no exact BDF term):** half-cycle capacity (BioLogic `Q charge`/`Capacity`);
per-mass/gravimetric capacity (Basytec `[Ah/kg]`); per-record ΔQ (BioLogic `dq`).

**Open question for you:** BioLogic single-cycle GITT can't separate *step* vs *half-cycle*
reset. Data says half-cycle (resets only at polarity flip, holds through rests/steps); ontology
note says EC-Lab `Q charge` is step-level. A multi-cycle EC-Lab file would settle it.

Neware xlsx `Chg./DChg. Cap.` step-vs-test level also needs a multi-step Neware export
(this sample is one charge step).

**Not verifiable here:** Arbin (no Zenodo dataset).
